# 02 — Model Training & Ablation

Trains and compares three architectures on the CGM prediction task:
1. **Naive baseline** — last observed value held constant
2. **Linear baseline** — flattened window → linear layer
3. **LSTM** — 2-layer LSTM with dropout
4. **Temporal CNN** — dilated causal residual blocks

The ablation table shows MARD and RMSE at both 30-min and 60-min horizons.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from data.sample.generate_synthetic import simulate_cgm
from src.models import GlucoseLSTM, GlucoseTemporalCNN, LinearBaseline, NaiveLastValueBaseline
from src.preprocessing import CGMProcessor
from src.evaluation import compute_all_metrics, clarke_error_grid
from src.visualization import plot_learning_curves, plot_ablation_table

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else
                       'cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Prepare data

In [ ]:
df_raw = simulate_cgm(days=30, seed=42)
processor = CGMProcessor(seq_len=24, pred_horizons=[6, 12])
df = processor.clean(df_raw)

n = len(df)
train_df = df.iloc[:int(n * 0.7)]
val_df   = df.iloc[int(n * 0.7):int(n * 0.85)]
test_df  = df.iloc[int(n * 0.85):]

processor.fit_scaler(train_df)
X_train, y_train = processor.make_windows(train_df)
X_val,   y_val   = processor.make_windows(val_df)
X_test,  y_test  = processor.make_windows(test_df)

def to_loader(X, y, batch_size=32, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, shuffle=True)
val_loader   = to_loader(X_val,   y_val)
test_loader  = to_loader(X_test,  y_test)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## 2. Training loop

In [ ]:
def train(model, n_epochs=60, patience=10, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    criterion = nn.MSELoss()
    train_losses, val_losses = [], []
    best_val, patience_ctr = float('inf'), 0
    best_state = None

    for epoch in range(1, n_epochs + 1):
        model.train()
        tl = sum(criterion(model(xb.to(device)), yb.to(device)).item()
                 for xb, yb in train_loader) / len(train_loader)
        model.eval()
        with torch.no_grad():
            vl = sum(criterion(model(xb.to(device)), yb.to(device)).item()
                     for xb, yb in val_loader) / len(val_loader)
        train_losses.append(tl); val_losses.append(vl)
        sched.step(vl)
        if vl < best_val:
            best_val, patience_ctr = vl, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'  Early stop @ epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return model, train_losses, val_losses

## 3. Train all models

In [ ]:
print('Training LSTM...')
lstm, lstm_tl, lstm_vl = train(GlucoseLSTM(hidden_size=64, num_layers=2, num_horizons=2))

print('Training Temporal CNN...')
tcn, tcn_tl, tcn_vl = train(GlucoseTemporalCNN(num_channels=[32, 64, 64], num_horizons=2))

print('Training Linear baseline...')
lin, lin_tl, lin_vl = train(LinearBaseline(seq_len=24, num_horizons=2))

## 4. Learning curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (tl, vl, name) in zip(axes, [
    (lstm_tl, lstm_vl, 'LSTM'),
    (tcn_tl, tcn_vl, 'Temporal CNN'),
    (lin_tl, lin_vl, 'Linear Baseline'),
]):
    ax.plot(tl, label='Train', color='#3498db')
    ax.plot(vl, label='Val', color='#e74c3c')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Learning Curves — All Models', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/plots/02_learning_curves.png', bbox_inches='tight')
plt.show()

## 5. Ablation results table

In [ ]:
def evaluate(model, label):
    model.eval().to(device)
    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()
    results = {}
    for i, h_min in enumerate([30, 60]):
        t = processor.inverse_transform(y_test[:, i])
        p = processor.inverse_transform(preds[:, i])
        results.update(compute_all_metrics(t, p, h_min))
    zones, _ = clarke_error_grid(
        processor.inverse_transform(y_test[:, 0]),
        processor.inverse_transform(preds[:, 0])
    )
    results['zone_A_%'] = zones['A']
    return results

naive = NaiveLastValueBaseline()
ablation = {
    'Naive (last value)': evaluate(naive, 'Naive'),
    'Linear':             evaluate(lin,   'Linear'),
    'LSTM':               evaluate(lstm,  'LSTM'),
    'Temporal CNN':       evaluate(tcn,   'Temporal CNN'),
}

import pandas as pd
abl_df = pd.DataFrame(ablation).T.round(2)
print(abl_df.to_string())

fig = plot_ablation_table(ablation, save_path='../results/plots/02_ablation_table.png')
plt.show()